# Attention as a Weighted Average
## A kernel tutorial for data scientists
Agus Sudjianto and Ming Yuan

An attention head computes a weighted average. We start with three observations, name the parts of that calculation, and then learn how a network can choose the weights and the values it averages.

This notebook connects attention to kernel smoothing, uses residual averages to improve a regression fit, and trains a small attention network. KRR and PSD kernels serve as comparisons. See [README.md](README.md) for installation and run instructions.

Run all cells from the top. All calculations use the CPU. Synthetic examples and training need no downloads. California Housing runs only if already cached. Figures and `results.json` are saved under `outputs/` in the working directory, or under `ATTENTION_TUTORIAL_OUTPUT_DIR` if set. The training experiment is a small demonstration, not a benchmark.

In [ ]:
from pathlib import Path
import copy, json, os, platform, time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch
from torch import nn
import sklearn
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

# Outputs are relative to the working directory unless explicitly configured.
OUTPUT = Path(os.environ.get('ATTENTION_TUTORIAL_OUTPUT_DIR', 'outputs')).expanduser()
FIG = OUTPUT / 'figures'
FIG.mkdir(parents=True, exist_ok=True)
thread_limits = threadpool_limits(limits=2)
torch.set_num_threads(2)
torch.set_default_dtype(torch.float64)
plt.rcParams.update({'figure.dpi': 115, 'axes.spines.top': False, 'axes.spines.right': False})
versions = dict(python=platform.python_version(), numpy=np.__version__, torch=torch.__version__,
                matplotlib=matplotlib.__version__, sklearn=sklearn.__version__)
results = {'versions': versions}
started = time.time()
print(versions)

def savefig(name):
    plt.savefig(FIG / (name + '.pdf'), bbox_inches='tight')
    plt.savefig(FIG / (name + '.png'), bbox_inches='tight', dpi=150)
    plt.show()

def rmse(y, pred):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(pred))**2)))

## 0. One query, three values, one average

Suppose three observations have labels **10, 20, and 40**. Their similarities to a new input are **6, 3, and 1**. Divide by their sum to obtain weights **0.6, 0.3, and 0.1**. The prediction is **16**.

Here is the attention vocabulary:

| Term | Role |
|---|---|
| Query | Represents the input asking for a prediction. |
| Key | Represents an observation used for comparison. |
| Score | Compares the query with a key. |
| Softmax | Exponentiates scores and normalizes them to sum to one. |
| Value | Contains the number or vector to be averaged. |
| Head output | Adds the weighted values. |

The kernel supplies similarities; normalized similarities supply weights. **The kernel is the similarity rule, not the weighted average itself.** With scores equal to the logarithms of our similarities, softmax gives exactly the weights above. If we change only the values, the weights stay the same but the answer changes.

In [ ]:
similarities = np.array([6., 3., 1.])
values = np.array([10., 20., 40.])
normalized = similarities / similarities.sum()
scores = np.log(similarities)
exp_scores = np.exp(scores - scores.max())
attention_weights = exp_scores / exp_scores.sum()
np.testing.assert_allclose(attention_weights, normalized)
np.testing.assert_allclose(attention_weights @ values, 16.)
print('Weights:', attention_weights)
print('Weighted label average:', attention_weights @ values)
print('Same weights, different values:', attention_weights @ np.array([1., -2., 3.]))

## 1. A curve, a context, and new inputs

The **context** contains the inputs and labels available at prediction time. Query inputs are the locations where we want predictions. Query labels are used only to measure error.

We draw context, validation, and test observations separately. Validation chooses bandwidth, ridge penalty, and depth. Test labels do not enter those choices. The noise-free curve is available here because this is a simulation; we use it for the plot, not for tuning.

In [ ]:
rng = np.random.default_rng(17)
def truth(x):
    return np.sin(2.5*x[:, 0]) + 0.35*np.cos(5*x[:, 0])
def sample(n):
    x = rng.uniform(-2, 2, size=(n, 1))
    return x, truth(x) + rng.normal(0, 0.18, n)
X, y = sample(48)
Xval, yval = sample(160)
Xtest, ytest = sample(240)
Xgrid = np.linspace(-2.2, 2.2, 400)[:, None]
ybar = y.mean()
yc = y - ybar
plt.figure(figsize=(7, 3))
plt.scatter(X[:, 0], y, s=20, label='Context labels')
plt.plot(Xgrid[:, 0], truth(Xgrid), color='black', label='Noise-free curve')
plt.xlabel('Input'); plt.ylabel('Response'); plt.legend()
savefig('data')

## 2. Let a kernel choose the weights

Use the Gaussian similarity $k(x,z)=\exp(-\|x-z\|^2/T)$. Nearby inputs get more weight. The parameter $T>0$ sets the distance scale; there is no extra factor of two in our convention.

Normalize the similarities from each query to the context and average labels:
$$Q_{ji}=\frac{k(x_j^q,x_i)}{\sum_a k(x_j^q,x_a)},\qquad \hat y_q=Qy.$$
This is NW regression. Centering the labels gives the same prediction: $\hat y_q=\bar y+Qy_c$, where $y_c=y-\bar y$.

For comparison, KRR uses the same kernel but solves for coefficients:
$$(K+\lambda I)\alpha=y_c,\qquad \hat y_q=\bar y+K_q\alpha.$$
Here $K$ contains context similarities, $K_q$ contains query-to-context similarities, and $\lambda$ is the diagonal penalty in the solve. KRR uses fitted coefficients rather than the labels and does not normalize the query similarities.

The Gaussian kernel is positive and PSD. These are different properties: positive entries allow normalized averaging, while PSD means $a^TKa\geq0$ for every vector $a$. We will examine why PSD matters for repeated corrections later.

In [ ]:
def logits(a, b, T):
    return -np.sum((a[:, None, :] - b[None, :, :])**2, axis=-1) / T

def softmax_rows(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def kernel(a, b, T):
    return np.exp(logits(a, b, T))

def weights(a, b, T):
    # Subtracting the row maximum prevents exponential underflow.
    return softmax_rows(logits(a, b, T))

def krr(a, b, labels, T, lam):
    mean = labels.mean()
    alpha = np.linalg.solve(kernel(a, a, T) + lam*np.eye(len(a)), labels-mean)
    return mean + kernel(b, a, T) @ alpha

T = 0.5  # fixed for the explanatory figures
K = kernel(X, X, T)
S = weights(X, X, T)
Q = weights(Xgrid, X, T)
assert np.linalg.eigvalsh(K).min() > -1e-10
np.testing.assert_allclose(S.sum(axis=1), 1)
np.testing.assert_allclose(ybar + Q @ yc, Q @ y)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
order = np.argsort(X[:, 0])
im = ax[0].imshow(S[np.ix_(order, order)], aspect='auto', cmap='viridis')
fig.colorbar(im, ax=ax[0], label='Attention weight')
ax[0].set(xlabel='Context key (sorted)', ylabel='Context query (sorted)')
ax[1].scatter(X[:, 0], y, s=12, color='gray')
ax[1].plot(Xgrid[:, 0], truth(Xgrid), 'k--', label='Truth')
ax[1].plot(Xgrid[:, 0], krr(X, Xgrid, y, T, .1), label='KRR, penalty 0.1')
ax[1].plot(Xgrid[:, 0], ybar+Q@yc, label='NW')
ax[1].set(xlabel='Input', ylabel='Prediction'); ax[1].legend()
plt.tight_layout(); savefig('krr_nw')

## 3. Write NW as attention

Gaussian attention can be computed with dot products and a key bias:
$$-\|x-z\|^2/T=-\|x\|^2/T+2x^Tz/T-\|z\|^2/T.$$
The first term is constant across keys and cancels in the softmax. The last term depends on the key and must stay. Ordinary dot-product attention without this bias generally gives different weights.

In this cell, queries and keys contain the input coordinates, and values contain centered labels. We compare two independent ways of forming the scores.

**Self-attention** uses the context as both queries and keys, producing a fitted value at each context input. **Cross-attention** uses new inputs as queries and the context as keys. Both perform the same weighted average. Several heads compute several averages with different maps; an output projection then combines them. That combination need not remain a convex average of the original labels.

In [ ]:
dot_scores = 2 * Xgrid @ X.T / T - np.sum(X**2, axis=1)[None, :] / T
Qdot = softmax_rows(dot_scores)
np.testing.assert_allclose(Qdot, Q, atol=2e-14)
np.testing.assert_allclose(ybar + Qdot @ yc, ybar + Q @ yc, atol=2e-14)
print('Gaussian scores and biased dot-product scores agree.')

## 4. Use the next layer to correct errors

Start with zero centered predictions $f_c$ at the context and $f_q$ at queries. One layer computes
$$r=y_c-f_c,\quad f_c\leftarrow f_c+\eta Sr,\quad f_q\leftarrow f_q+\eta Qr.$$
Both updates use the **same old residual**. Updating the context first and then recomputing the residual for the query would shift the query calculation by one layer.

At unit step size, the first layer is NW. The second query prediction is $\bar y+Q(2I-S)y_c$. Residual correction differs from repeatedly smoothing the previous prediction, which gradually removes variation.

In [ ]:
def iterate(S, Q, centered_labels, depth, eta=1.0):
    fc = np.zeros_like(centered_labels)
    fq = np.zeros(Q.shape[0])
    context_path, query_path = [], []
    for _ in range(depth):
        residual = centered_labels - fc  # read before either update
        fc = fc + eta * (S @ residual)
        fq = fq + eta * (Q @ residual)
        context_path.append(fc.copy())
        query_path.append(fq.copy())
    return np.array(context_path), np.array(query_path)

fc_path, fq_path = iterate(S, Q, yc, 128)
np.testing.assert_allclose(fq_path[0], Q @ yc, atol=1e-13)
np.testing.assert_allclose(fq_path[1], Q @ (2*np.eye(len(X))-S) @ yc, atol=1e-13)
# An independent matrix-power expression checks deeper updates.
v = sum((np.linalg.matrix_power(np.eye(len(X))-S, j) @ yc for j in range(8)))
np.testing.assert_allclose(fq_path[7], Q @ v, atol=1e-12)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for depth in [1, 4, 16, 128]:
    ax[0].plot(Xgrid[:, 0], ybar+fq_path[depth-1], label=f'{depth} layers')
ax[0].scatter(X[:, 0], y, s=10, color='gray'); ax[0].legend()
ax[0].set(xlabel='Input', ylabel='Prediction')
ax[1].plot(np.arange(1, 129), np.sqrt(np.mean((yc-fc_path)**2, axis=1)))
ax[1].set(xlabel='Layers', ylabel='Context RMSE', xscale='log')
plt.tight_layout(); savefig('depth')

## 5. Choose depth using validation labels

A smaller context error does not establish better prediction on new observations. We select depth and bandwidth using validation error. KRR gets its own bandwidth and penalty search. After selection, all methods still fit only the original context; we do not add validation labels to it.

The next plot holds the bandwidth fixed to show the effect of depth. The table then tunes each method separately. This is one small draw of data, not evidence that one estimator is generally best.

In [ ]:
depths = [1, 2, 4, 8, 16, 32, 64, 128]
temperatures = [0.08, 0.2, 0.5, 1.0, 2.0]
penalties = [0.001, 0.01, 0.1, 1.0]
_, val_path = iterate(S, weights(Xval, X, T), yc, 128)
plt.figure(figsize=(6, 3))
plt.plot(np.arange(1, 129), [rmse(yval, ybar+p) for p in val_path], label='Validation')
plt.plot(np.arange(1, 129), [rmse(y, ybar+p) for p in fc_path], label='Context')
plt.xlabel('Layers'); plt.ylabel('RMSE'); plt.xscale('log'); plt.legend()
savefig('validation')

nw_candidates, it_candidates, kr_candidates = [], [], []
for temp in temperatures:
    _, path = iterate(weights(X, X, temp), weights(Xval, X, temp), yc, max(depths))
    nw_candidates.append((rmse(yval, ybar+path[0]), temp))
    it_candidates.extend((rmse(yval, ybar+path[d-1]), temp, d) for d in depths)
    kr_candidates.extend((rmse(yval, krr(X, Xval, y, temp, lam)), temp, lam) for lam in penalties)
_, tn = min(nw_candidates)
_, ti, di = min(it_candidates)
_, tk, lk = min(kr_candidates)
synthetic = {
    'NW': {'T': tn, 'test_rmse': rmse(ytest, ybar+weights(Xtest, X, tn)@yc)},
    'Iterated NW': {'T': ti, 'depth': di, 'test_rmse': rmse(ytest, ybar+iterate(weights(X,X,ti), weights(Xtest,X,ti), yc, di)[1][-1])},
    'KRR': {'T': tk, 'penalty': lk, 'test_rmse': rmse(ytest, krr(X,Xtest,y,tk,lk))}}
results['synthetic'] = synthetic
print(json.dumps(synthetic, indent=2))

## 6. What PSD gives us, and what changes without it

For a positive, symmetric PSD kernel, $S=D^{-1}K$ is similar to the symmetric matrix $A=D^{-1/2}KD^{-1/2}$. Its eigenvalues lie in $[0,1]$. With unit residual steps, an eigencomponent of size $s$ is fitted by the fraction $1-(1-s)^L$ after $L$ layers. Small eigenvalues are fitted slowly. This statement uses a fixed kernel and the degree-weighted coordinates, not arbitrary changing attention matrices.

Positive attention weights alone do not give this guarantee. The example below is symmetric but **not PSD**, showing that asymmetry is not needed for failure. It is a valid softmax matrix: use its entrywise logarithm as logits. Its negative eigenvalue makes the residual grow. An asymmetric example fails too. These are counterexamples, not claims that every non-PSD smoother diverges.

In [ ]:
degrees = K.sum(axis=1)
A = K / np.sqrt(degrees[:, None]*degrees[None, :])
eig = np.linalg.eigvalsh(A)
assert eig.min() > -1e-12 and eig.max() <= 1+1e-12
Sbad = np.array([[0.1, 0.9], [0.9, 0.1]])
Sasym = np.array([[0.1, 0.9], [0.8, 0.2]])
np.testing.assert_allclose(softmax_rows(np.log(Sbad)), Sbad)
r0 = np.array([1., -1.])
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
s = np.linspace(0, 1, 300)
for d in [1, 4, 16, 64]:
    ax[0].plot(s, 1-(1-s)**d, label=f'{d} layers')
ax[0].set(xlabel='Eigenvalue', ylabel='Fraction fitted'); ax[0].legend()
for name, mat in [('Symmetric, non-PSD', Sbad), ('Asymmetric', Sasym)]:
    residuals = [np.linalg.norm(np.linalg.matrix_power(np.eye(2)-mat, l)@r0) for l in range(16)]
    ax[1].semilogy(range(16), residuals, label=name)
ax[1].set(xlabel='Layers', ylabel='Residual norm'); ax[1].legend()
plt.tight_layout(); savefig('stability')
print('Non-PSD eigenvalues:', np.linalg.eigvals(Sbad))
print('Asymmetric eigenvalues:', np.linalg.eigvals(Sasym))

### Interpolating the context still does not make NW equal to KRR

For an invertible $K$, the infinite-depth NW limit is
$$\bar y+\frac{k_x^T K^{-1}D y_c}{d(x)},$$
where $D$ holds context row sums and $d(x)$ is the query row sum. Ridgeless KRR is $\bar y+k_x^TK^{-1}y_c$. Both fit context labels exactly, but their query predictions generally differ. We use a separate, well-conditioned example so this check is not dominated by numerical inversion of an almost singular Gaussian matrix.

In [ ]:
Xi = np.array([[-1.5], [-0.4], [0.2], [1.4]])
yi = np.array([1.0, -0.5, 0.8, -0.2])
mi, ci = yi.mean(), yi-yi.mean()
Ki = kernel(Xi, Xi, 0.5)
diag = Ki.sum(axis=1)
Si = weights(Xi, Xi, 0.5)
Xqi = np.linspace(-2, 2, 100)[:, None]
Qi = weights(Xqi, Xi, 0.5)
limit = mi + Qi @ np.linalg.solve(Si, ci)
ridgeless = mi + kernel(Xqi, Xi, 0.5) @ np.linalg.solve(Ki, ci)
np.testing.assert_allclose(mi + Si @ np.linalg.solve(Si, ci), yi)
np.testing.assert_allclose(mi + Ki @ np.linalg.solve(Ki, ci), yi)
assert np.max(np.abs(limit-ridgeless)) > 0.01
_, ip = iterate(Si, Qi, ci, 300)
np.testing.assert_allclose(mi+ip[-1], limit, atol=1e-10)
print('Condition number:', np.linalg.cond(Ki))
print('Largest query difference:', np.max(np.abs(limit-ridgeless)))

## 7. Put the calculation into trainable attention layers

Each token holds three numbers: $(x,y_c,f)$. Context tokens have a centered label; query tokens have a zero placeholder. Only context tokens supply keys and values. All tokens may supply queries.

At the NW initialization, queries and keys read $x$, values read $y_c-f$, and attention writes only to $f$. No layer normalization or feed-forward network is included. Both context and query outputs are formed from the old context state before either state changes.

After training, query/key projections may read other channels and the value map may cease to be a residual. This is a small attention architecture designed to expose the calculation, not a full implementation of a standard transformer. The output remains restricted to the prediction channel.

In [ ]:
class SmoothingLayer(nn.Module):
    def __init__(self, nw_init=True, T=0.5):
        super().__init__()
        self.q = nn.Linear(3, 2, bias=False)
        self.k = nn.Linear(3, 2, bias=False)
        self.v = nn.Linear(3, 1, bias=False)
        self.T = T
        if nw_init:
            with torch.no_grad():
                self.q.weight.zero_(); self.k.weight.zero_()
                self.q.weight[0, 0] = 1; self.k.weight[0, 0] = 1
                self.v.weight.copy_(torch.tensor([[0., 1., -1.]]))

    def forward(self, context, query):
        old_keys, old_values = self.k(context), self.v(context)
        def update(tokens):
            projected = self.q(tokens)
            scores = -((projected[:, :, None, :] - old_keys[:, None, :, :])**2).sum(-1)/self.T
            correction = scores.softmax(-1) @ old_values
            return torch.cat([tokens[..., :2], tokens[..., 2:3]+correction], dim=-1)
        return update(context), update(query)

class SmoothingNetwork(nn.Module):
    def __init__(self, depth=4, nw_init=True):
        super().__init__()
        self.layers = nn.ModuleList([SmoothingLayer(nw_init) for _ in range(depth)])
    def forward(self, xc, labels, xq):
        mean = labels.mean(dim=1, keepdim=True)
        centered = labels-mean
        context = torch.stack([xc, centered, torch.zeros_like(xc)], dim=-1)
        query = torch.stack([xq, torch.zeros_like(xq), torch.zeros_like(xq)], dim=-1)
        for layer in self.layers:
            context, query = layer(context, query)
        return mean + query[..., 2]

net = SmoothingNetwork()
with torch.no_grad():
    pred = net(torch.tensor(X[:, 0])[None], torch.tensor(y)[None], torch.tensor(Xgrid[:, 0])[None])[0].numpy()
np.testing.assert_allclose(pred, ybar+fq_path[3], atol=1e-12)
# Context order and query batch composition must not change the prediction.
perm = np.random.default_rng(5).permutation(len(X))
with torch.no_grad():
    reordered = net(torch.tensor(X[perm, 0])[None], torch.tensor(y[perm])[None], torch.tensor(Xgrid[:, 0])[None])[0].numpy()
    alone = net(torch.tensor(X[:, 0])[None], torch.tensor(y)[None], torch.tensor(Xgrid[:1, 0])[None])[0].numpy()
np.testing.assert_allclose(reordered, pred, atol=1e-12)
np.testing.assert_allclose(alone, pred[:1], atol=1e-12)
print('Token network equals four residual NW updates; masking and ordering checks passed.')

## 8. Train on many functions

A single fixed target lets a network learn that target without needing the context labels. We instead draw a new function in every episode:
$$ f(x)=a\sin(2x)+b\cos(3x)+cx+d.$$
The coefficients vary by episode. Each episode has 20 labeled context points and 20 query points. The network receives no query labels. It sees those labels only through the training loss.

We compare NW and random initialization using the same architecture, training episodes, optimizer, and checkpoint rule. Each run has 250 updates; validation chooses a checkpoint every 25 updates, including step zero. Three seeds provide a small check of variability. Test episodes use new coefficients and observations from the same distribution. This does **not** test transfer to a new function family or reproduce the held-out-target experiment in larger benchmark studies.

In [ ]:
def episodes(seed, batch=32, n=20, m=20):
    gen = np.random.default_rng(seed)
    xc = gen.uniform(-2, 2, (batch, n))
    xq = gen.uniform(-2, 2, (batch, m))
    coeff = gen.normal(size=(batch, 4)) * np.array([1., .5, .3, .5])
    def f(x):
        return coeff[:, 0:1]*np.sin(2*x)+coeff[:, 1:2]*np.cos(3*x)+coeff[:, 2:3]*x+coeff[:, 3:4]
    yc_ep = f(xc)+gen.normal(0, .12, xc.shape)
    yq_ep = f(xq)+gen.normal(0, .12, xq.shape)
    return tuple(torch.tensor(a) for a in (xc, yc_ep, xq, yq_ep))

validation = episodes(800001, batch=64)
test = episodes(900001, batch=128)

def mse_model(model, episode):
    xc, labels, xq, target = episode
    return ((model(xc, labels, xq)-target)**2).mean()

def train_one(seed, nw_init):
    torch.manual_seed(seed)
    model = SmoothingNetwork(nw_init=nw_init)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    with torch.no_grad():
        best = float(mse_model(model, validation))
    state, best_step = copy.deepcopy(model.state_dict()), 0
    trace = [(0, best)]
    for step in range(1, 251):
        episode = episodes(seed*10000+step, batch=16)
        optimizer.zero_grad()
        loss = mse_model(model, episode)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if step % 25 == 0:
            with torch.no_grad():
                val_loss = float(mse_model(model, validation))
            trace.append((step, val_loss))
            if val_loss < best:
                best, state, best_step = val_loss, copy.deepcopy(model.state_dict()), step
    model.load_state_dict(state)
    model.eval()
    return model, trace, best_step

models, traces, training_rows = {}, {}, []
for seed in [1, 2, 3]:
    for name, init in [('NW initialization', True), ('Random initialization', False)]:
        model, trace, step = train_one(seed, init)
        models[(name, seed)], traces[(name, seed)] = model, trace
        with torch.no_grad():
            test_rmse = float(mse_model(model, test).sqrt())
        training_rows.append(dict(seed=seed, initialization=name, selected_step=step, test_rmse=test_rmse))
with torch.no_grad():
    baseline_rmse = float(mse_model(SmoothingNetwork(), test).sqrt())
results['training'] = {'runs': training_rows, 'untrained_iterated_nw_rmse': baseline_rmse,
                       'steps': 250, 'depth': 4, 'seeds': [1,2,3]}
print('Untrained four-layer NW:', baseline_rmse)
for row in training_rows:
    print(row)
plt.figure(figsize=(7, 3))
for name in ['NW initialization', 'Random initialization']:
    curves = np.array([traces[(name, seed)] for seed in [1,2,3]])
    plt.plot(curves[0,:,0], np.sqrt(curves[:,:,1]).mean(axis=0), label=name)
plt.xlabel('Training updates'); plt.ylabel('Mean validation RMSE'); plt.legend()
savefig('training')

### Check whether predictions use the context labels

Permuting labels within each context preserves their mean but breaks the input-label pairing. We report prediction error after this change and the RMS change in predictions. A change shows dependence on the pairing; it does not by itself prove useful reasoning. The ordinary test error is needed as well.

We also compare predictions with the untrained NW network. A nonzero difference shows that training changed the function computed by the model. It does not identify which parameter change caused the difference.

In [ ]:
xc, labels, xq, target = test
shuffle_rng = np.random.default_rng(300001)
permutations = torch.tensor(np.stack([shuffle_rng.permutation(labels.shape[1]) for _ in range(len(labels))]))
shuffled_labels = labels.gather(1, permutations)
diagnostics = []
with torch.no_grad():
    reference = SmoothingNetwork()(xc, labels, xq)
    for (name, seed), model in models.items():
        original = model(xc, labels, xq)
        shuffled = model(xc, shuffled_labels, xq)
        diagnostics.append(dict(initialization=name, seed=seed,
            test_rmse=float(((original-target)**2).mean().sqrt()),
            shuffled_rmse=float(((shuffled-target)**2).mean().sqrt()),
            prediction_change=float(((shuffled-original)**2).mean().sqrt()),
            departure_from_nw=float(((original-reference)**2).mean().sqrt())))
results['diagnostics'] = diagnostics
for name in ['NW initialization', 'Random initialization']:
    rows = [d for d in diagnostics if d['initialization']==name]
    print(name, {k: round(float(np.mean([r[k] for r in rows])), 4)
                 for k in ['test_rmse','shuffled_rmse','prediction_change','departure_from_nw']})

## 9. A tabular example: California Housing

This section runs only when scikit-learn's California Housing data are already cached. If they are absent, the cell prints a message and leaves the synthetic examples intact. To obtain the data separately, call `fetch_california_housing()` in a session where you want to allow a download, then rerun this notebook.

We use 192 context rows, 192 validation rows, and 384 test rows selected without overlap. Input scaling is fitted on context rows only. Labels are standardized using the context mean and standard deviation; the table reports RMSE in those units. Each method selects its parameters on validation rows. This small random split illustrates the workflow; it is not a spatial generalization study, and its scores are not comparable to the larger experiments in larger benchmark studies.

Set `ATTENTION_TUTORIAL_DATA_DIR` to use a custom cache directory. Set `ATTENTION_TUTORIAL_SKIP_HOUSING=1` to skip this section even when a cache exists. Neither setting is required.

In [ ]:
housing = {'status': 'skipped: California Housing is not cached'}
try:
    if os.environ.get('ATTENTION_TUTORIAL_SKIP_HOUSING') == '1':
        raise OSError('Housing example disabled')
    data = fetch_california_housing(
        data_home=os.environ.get('ATTENTION_TUTORIAL_DATA_DIR'),
        download_if_missing=False)
except OSError:
    print(housing['status'])
else:
    ids = np.random.default_rng(44).permutation(len(data.target))[:768]
    c, v, t = ids[:192], ids[192:384], ids[384:]
    scaler = StandardScaler().fit(data.data[c])
    hx, hv, ht = (scaler.transform(data.data[idx]) for idx in [c,v,t])
    ym, ys = data.target[c].mean(), data.target[c].std()
    hy, hyv, hyt = ((data.target[idx]-ym)/ys for idx in [c,v,t])
    candidates = {'NW': [], 'Iterated NW': [], 'KRR': []}
    for temp in [1., 4., 16., 64.]:
        hs, hq = weights(hx,hx,temp), weights(hv,hx,temp)
        _, hp = iterate(hs,hq,hy-hy.mean(),32)
        candidates['NW'].append((rmse(hyv,hy.mean()+hp[0]),temp,1))
        candidates['Iterated NW'].extend((rmse(hyv,hy.mean()+hp[d-1]),temp,d) for d in [1,2,4,8,16,32])
        candidates['KRR'].extend((rmse(hyv,krr(hx,hv,hy,temp,lam)),temp,lam) for lam in [.01,.1,1.,10.])
    rows = {}
    for name, choices in candidates.items():
        score,temp,parameter = min(choices)
        if name == 'KRR':
            prediction = krr(hx,ht,hy,temp,parameter)
        else:
            prediction = hy.mean()+iterate(weights(hx,hx,temp),weights(ht,hx,temp),hy-hy.mean(),int(parameter))[1][-1]
        rows[name] = dict(T=temp, parameter=parameter, validation_rmse=score, test_rmse=rmse(hyt,prediction))
    housing = dict(status='completed', context=192, validation=192, test=384,
                   target_units='context standard deviations', rows=rows)
    print(json.dumps(housing, indent=2))
results['housing'] = housing

## 10. What to change next

Try a smaller bandwidth and inspect which context points receive weight. Increase the noise and see whether validation selects fewer layers. Remove the key bias in Section 3 and measure the difference from Gaussian NW. In Section 7, keep the context prediction fixed and observe why the queries no longer receive successive residual corrections.

For your own tabular data, split observations before fitting preprocessing. Keep query labels out of token states. Select bandwidth, depth, and training checkpoints using validation data. Compare with KRR on the same inputs. If you learn separate query and key maps, inspect residual norms as well as prediction error.

The examples are deliberately small enough to inspect. They establish the averaging identities and illustrate training; they do not establish performance on other datasets or function families.

In [ ]:
results['elapsed_seconds'] = time.time()-started
(OUTPUT / 'results.json').write_text(json.dumps(results, indent=2)+chr(10))
print(f"Saved results.json and figures in the output directory. Runtime: {results['elapsed_seconds']:.1f} seconds.")